In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

import requests
from bs4 import BeautifulSoup

def fetch_website_contents(url: str) -> str:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.text

def fetch_website_links(url: str):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    links = []
    for a in soup.find_all("a", href=True):
        links.append(a["href"])
    
    return links

print("Tudo pronto 🚀")

In [ ]:
from openai import OpenAI

MODEL = "llama3.2"

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
    
)

In [ ]:
links = fetch_website_links("https://converse.com.br/")
links

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.

Select the most relevant links for a company brochure (e.g., About, Careers, Company pages).

You MUST follow these rules:
- Return ONLY valid JSON
- Do NOT include any text before or after the JSON
- Use EXACTLY this format:

{
  "links": [
    {"type": "about", "url": "https://example.com/about"},
    {"type": "careers", "url": "https://example.com/careers"}
  ]
}
"""

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://converse.com.br/"))

In [ ]:
def select_relevant_links(url):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [ ]:
select_relevant_links("https://converse.com.br/")

In [ ]:
import json
import re

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ]
        
    )
    
    result = response.choices[0].message.content
    print("Resposta do modelo:\n", result) 
    
    
    try:
        data = json.loads(result)
    except:
        match = re.search(r"\{.*\}", result, re.DOTALL)
        data = json.loads(match.group()) if match else {}
    
    links = data.get("links", [])
    
    print(f"Found {len(links)} relevant links")
    return links

In [ ]:
select_relevant_links("https://converse.com.br/")

Segundo passo

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://converse.com.br"))

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("Converse", "https://converse.com.br")

In [ ]:
def create_brochure(company_name, url):
    response = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("Converse", "https://converse.com.br")

In [ ]:

def stream_brochure(company_name, url):
    stream = client.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Converse", "https://converse.com.br")

BrochureWithGradio

In [14]:
import gradio as gr

In [15]:
brochure_system_prompt = """
You are a professional marketing assistant.
Create a concise, attractive brochure for the company.
Use markdown formatting.
"""

In [16]:
def get_brochure_user_prompt(company_name, url):
    return f"Create a brochure for {company_name}. Website: {url}"

In [17]:
def create_brochure_stream(company_name, url):
    messages = [
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
    ]

    stream = client.chat.completions.create(
        model="llama3.2",
        messages=messages,
        stream=True
    )

    result = ""

    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            result += delta
            yield result


interface = gr.Interface(
    fn=create_brochure_stream,
    inputs=[
        gr.Textbox(label="Nome da empresa"),
        gr.Textbox(label="URL da empresa")
    ],
    outputs="markdown",
    title="Gerador de Brochure com IA",
    flagging_mode="never"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
